# Stage 1 Deliverables — Dataset Statistics

Produces `outputs/stage1_dataset_stats.xlsx` with one sheet per programming language.

Each sheet contains:
- **Dataset** name, total samples, vulnerable samples, safe samples, unique CWE count
- One column per unique CWE with its count per dataset
- CWE columns are colour-coded by granularity:
  - **Green + bold** → leaf CWE (most specific, no children in the CWE tree)
  - **Amber + bold** → non-leaf CWE (intermediate / umbrella node)
  - **Gray + bold** → CWE not found in the MITRE XML (unknown / malformed ID)

**Prerequisites:**
- Run `00_download_datasets.ipynb` first
- `data/cwec_latest.xml` must exist (cell below downloads it if missing)

In [1]:
import sys
from pathlib import Path

# Make project root importable
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW     = ROOT / 'data' / 'raw'
CWE_XML = ROOT / 'data' / 'cwec_latest.xml'
OUT_DIR = ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
XLSX_PATH = OUT_DIR / 'stage1_dataset_stats.xlsx'

print(f'Root:    {ROOT}')
print(f'Raw:     {RAW}')
print(f'CWE XML: {CWE_XML}')
print(f'Output:  {XLSX_PATH}')

Root:    C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion
Raw:     C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion\data\raw
CWE XML: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion\data\cwec_latest.xml
Output:  C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion\outputs\stage1_dataset_stats.xlsx


## 1. CWE XML — download if missing

MITRE distributes the full CWE catalogue as a zipped XML. Downloaded once to `data/cwec_latest.xml`.

In [2]:
import io, urllib.request, zipfile

CWE_ZIP_URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'

if CWE_XML.exists():
    print(f'CWE XML already present ({CWE_XML.stat().st_size / 1e6:.1f} MB) — skipping download.')
else:
    print('Downloading CWE XML from MITRE ...')
    with urllib.request.urlopen(CWE_ZIP_URL, timeout=60) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        xml_name = next(n for n in zf.namelist() if n.endswith('.xml'))
        CWE_XML.write_bytes(zf.read(xml_name))
    print(f'Saved: {CWE_XML} ({CWE_XML.stat().st_size / 1e6:.1f} MB)')

Saved: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion\data\cwec_latest.xml (16.1 MB)


## 2. Load CWE Navigator

Builds two helpers:
- `is_leaf(cwe_str)` — `True` if the CWE has no children in the MITRE tree
- `cwe_label(cwe_str)` — `'leaf'` | `'non-leaf'` | `'unknown'`

In [3]:
from ingestion.cwe_navigator import CWENavigator

print('Loading CWE Navigator ...')
nav = CWENavigator(str(CWE_XML))
print(f'Loaded: {len(nav.weaknesses):,} weaknesses, {len(nav.categories):,} categories, {len(nav.views):,} views')

# Build set of all IDs that appear as a parent of some other CWE
_parent_ids: set[str] = {p for parents in nav.child_of.values() for p in parents}

def _strip(cwe_str: str) -> str:
    """'CWE-119' -> '119'"""
    return cwe_str.removeprefix('CWE-').removeprefix('cwe-').strip()

def is_leaf(cwe_str: str) -> bool:
    """True if cwe_str is a known Weakness with no children."""
    num = _strip(cwe_str)
    return num in nav.weaknesses and num not in _parent_ids

def cwe_label(cwe_str: str) -> str:
    num = _strip(cwe_str)
    if num not in nav.weaknesses and num not in nav.categories:
        return 'unknown'
    return 'leaf' if is_leaf(cwe_str) else 'non-leaf'

# Quick sanity check
for sample_cwe in ['CWE-119', 'CWE-120', 'CWE-787', 'CWE-1000']:
    num = _strip(sample_cwe)
    name = nav.get_element_name(num)
    print(f'  {sample_cwe:12s}  {cwe_label(sample_cwe):8s}  {name}')

Loading CWE Navigator ...


Loaded: 969 weaknesses, 420 categories, 58 views
  CWE-119       non-leaf  Improper Restriction of Operations within the Bounds of a Memory Buffer
  CWE-120       non-leaf  Buffer Copy without Checking Size of Input ('Classic Buffer Overflow')
  CWE-787       non-leaf  Out-of-bounds Write
  CWE-1000      unknown   Unknown


## 3. Extract samples from each dataset

Datasets that have not been downloaded yet are skipped with a warning.

In [4]:
from ingestion.primevul   import extract_primevul
from ingestion.icvul      import extract_icvul
from ingestion.cvefixes   import extract_cvefixes
from ingestion.megavul    import extract_megavul
from ingestion.secvuleval import extract_secvuleval
from ingestion.schema     import FunctionSample

def _load_primevul():
    train = RAW / 'primevul_train.jsonl'
    test  = RAW / 'primevul_test.jsonl'
    if not train.exists() or not test.exists():
        raise FileNotFoundError(f'PrimeVul files not found in {RAW}')
    return extract_primevul(train) + extract_primevul(test)

def _load_icvul():
    icvul_dir = RAW / 'icvul'
    if not icvul_dir.exists():
        raise FileNotFoundError(f'ICVul directory not found: {icvul_dir}')
    funcs   = next(icvul_dir.rglob('function_info.csv'), None)
    mapping = next(icvul_dir.rglob('cve_fc_vcc_mapping.csv'), None)
    if funcs is None or mapping is None:
        raise FileNotFoundError(
            f'ICVul: could not find function_info.csv or cve_fc_vcc_mapping.csv under {icvul_dir}. '
            f'Files present: {[f.name for f in icvul_dir.rglob("*.csv")]}'
        )
    return extract_icvul(funcs.parent)

def _load_cvefixes():
    d = RAW / 'cvefixes'
    if not d.exists() or not any(d.glob('*.parquet')):
        raise FileNotFoundError(f'CVEfixes Parquet files not found in {d}')
    return extract_cvefixes(d)

def _load_megavul():
    d = RAW / 'megavul'
    if not d.exists() or not any(d.glob('*.parquet')):
        raise FileNotFoundError(f'MegaVul Parquet files not found in {d}')
    return extract_megavul(d)

def _load_secvuleval():
    data = RAW / 'secvuleval.csv'
    if not data.exists():
        raise FileNotFoundError(f'SecVulEval file not found: {data}')
    return extract_secvuleval(data)

LOADERS = {
    'PrimeVul':   _load_primevul,
    'ICVul':      _load_icvul,
    'CVEfixes':   _load_cvefixes,
    'MegaVul':    _load_megavul,
    'SecVulEval': _load_secvuleval,
}

collections: dict[str, list[FunctionSample]] = {}

for name, loader in LOADERS.items():
    try:
        samples = loader()
        collections[name] = samples
        print(f'  {name:12s}: {len(samples):>6,} samples loaded')
    except FileNotFoundError as e:
        print(f'  {name:12s}: SKIPPED — {e}')

print(f'\nDatasets available: {list(collections)}')

  PrimeVul    : 200,008 samples loaded


  ICVul       :      0 samples loaded


  CVEfixes    :  2,518 samples loaded


  MegaVul     :  1,224 samples loaded


  SecVulEval  : 25,440 samples loaded

Datasets available: ['PrimeVul', 'ICVul', 'CVEfixes', 'MegaVul', 'SecVulEval']


## 4. Compute statistics

In [5]:
from collections import Counter

# stats[dataset_name] = {
#   'total': int, 'vulnerable': int, 'safe': int,
#   'unique_cwes': int, 'cwe_counts': Counter
# }
stats: dict[str, dict] = {}

for name, samples in collections.items():
    vuln    = [s for s in samples if s.label == 1]
    safe    = [s for s in samples if s.label == 0]
    counter = Counter(cwe for s in vuln for cwe in s.cwes)
    stats[name] = {
        'total':       len(samples),
        'vulnerable':  len(vuln),
        'safe':        len(safe),
        'unique_cwes': len(counter),
        'cwe_counts':  counter,
    }

# Print summary
print(f'{"Dataset":<14} {"Total":>8} {"Vuln":>8} {"Safe":>8} {"Unique CWEs":>12}')
print('-' * 56)
for name, s in stats.items():
    print(f'{name:<14} {s["total"]:>8,} {s["vulnerable"]:>8,} {s["safe"]:>8,} {s["unique_cwes"]:>12,}')

# All unique CWEs across datasets, sorted by total count descending
global_counter: Counter = Counter()
for s in stats.values():
    global_counter.update(s['cwe_counts'])

ALL_CWES: list[str] = [cwe for cwe, _ in global_counter.most_common()]
print(f'\nTotal unique CWEs across all datasets: {len(ALL_CWES)}')

Dataset           Total     Vuln     Safe  Unique CWEs
--------------------------------------------------------
PrimeVul        200,008    4,834  195,174          121
ICVul                 0        0        0            0
CVEfixes          2,518    2,518        0          117
MegaVul           1,224    1,224        0           76
SecVulEval       25,440   10,998   14,442          231

Total unique CWEs across all datasets: 257


## 5. Build Excel report

Sheet **C_C++** layout:
```
| Dataset | Total | Vulnerable | Safe | Unique CWEs | CWE-119 | CWE-787 | ... |
```
CWE header cells are colour-coded:
- **Green** (#92D050) + bold → leaf node
- **Amber** (#FFC000) + bold → non-leaf (umbrella) node  
- **Gray**  (#BFBFBF) + bold → unknown / not in MITRE XML

In [6]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Colour palette ──────────────────────────────────────────────────────────
FILL_HEADER   = PatternFill('solid', fgColor='1F4E79')   # dark blue  — fixed cols header
FILL_LEAF     = PatternFill('solid', fgColor='92D050')   # green      — leaf CWE
FILL_NONLEAF  = PatternFill('solid', fgColor='FFC000')   # amber      — non-leaf CWE
FILL_UNKNOWN  = PatternFill('solid', fgColor='BFBFBF')   # gray       — unknown CWE
FILL_ROW_ODD  = PatternFill('solid', fgColor='F2F2F2')   # light gray — odd data rows
FILL_ZERO     = PatternFill('solid', fgColor='FFFFFF')   # white      — zero count

FONT_HEADER   = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
FONT_CWE      = Font(bold=True, color='000000', name='Calibri', size=10)
FONT_DATA     = Font(name='Calibri', size=10)
FONT_DATA_B   = Font(bold=True, name='Calibri', size=10)

ALIGN_CENTER  = Alignment(horizontal='center', vertical='center', wrap_text=True)
ALIGN_LEFT    = Alignment(horizontal='left',   vertical='center')

THIN = Side(style='thin', color='D3D3D3')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def _cwe_fill(cwe_str: str) -> PatternFill:
    label = cwe_label(cwe_str)
    return {'leaf': FILL_LEAF, 'non-leaf': FILL_NONLEAF, 'unknown': FILL_UNKNOWN}[label]

# ── Build workbook ───────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)  # remove default sheet

LANGUAGES = {'C_C++': list(stats.keys())}  # extend for future languages

for lang, dataset_names in LANGUAGES.items():
    if not dataset_names:
        continue

    ws = wb.create_sheet(title=lang)

    # CWEs for this language's datasets (ordered by global frequency)
    lang_counter: Counter = Counter()
    for dn in dataset_names:
        lang_counter.update(stats[dn]['cwe_counts'])
    lang_cwes = [cwe for cwe, _ in lang_counter.most_common()]

    # ── Fixed column headers ────────────────────────────────────────────────
    FIXED_COLS = ['Dataset', 'Total Samples', 'Vulnerable', 'Safe', 'Unique CWEs']
    N_FIXED = len(FIXED_COLS)

    for col_idx, label in enumerate(FIXED_COLS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=label)
        cell.fill   = FILL_HEADER
        cell.font   = FONT_HEADER
        cell.alignment = ALIGN_CENTER
        cell.border = BORDER

    # ── CWE column headers ──────────────────────────────────────────────────
    for cwe_idx, cwe in enumerate(lang_cwes, start=N_FIXED + 1):
        num = _strip(cwe)
        name = nav.get_element_name(num)
        label_str = cwe_label(cwe)
        # Header value: "CWE-119\nBuffer Overflow"
        header_val = f'{cwe}\n{name}' if name != 'Unknown' else cwe
        cell = ws.cell(row=1, column=cwe_idx, value=header_val)
        cell.fill      = _cwe_fill(cwe)
        cell.font      = FONT_CWE
        cell.alignment = ALIGN_CENTER
        cell.border    = BORDER

    # ── Data rows ───────────────────────────────────────────────────────────
    for row_idx, dn in enumerate(dataset_names, start=2):
        s = stats[dn]
        row_fill = FILL_ROW_ODD if row_idx % 2 == 1 else PatternFill()  # alternating

        fixed_vals = [dn, s['total'], s['vulnerable'], s['safe'], s['unique_cwes']]
        for col_idx, val in enumerate(fixed_vals, start=1):
            cell = ws.cell(row=row_idx, column=col_idx, value=val)
            cell.font      = FONT_DATA_B if col_idx == 1 else FONT_DATA
            cell.alignment = ALIGN_LEFT if col_idx == 1 else ALIGN_CENTER
            cell.fill      = row_fill
            cell.border    = BORDER

        for cwe_idx, cwe in enumerate(lang_cwes, start=N_FIXED + 1):
            count = s['cwe_counts'].get(cwe, 0)
            cell = ws.cell(row=row_idx, column=cwe_idx, value=count if count > 0 else None)
            cell.font      = FONT_DATA
            cell.alignment = ALIGN_CENTER
            cell.fill      = row_fill
            cell.border    = BORDER

    # ── Column widths ────────────────────────────────────────────────────────
    ws.column_dimensions['A'].width = 18   # Dataset
    for c in range(2, N_FIXED + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    for c in range(N_FIXED + 1, N_FIXED + len(lang_cwes) + 1):
        ws.column_dimensions[get_column_letter(c)].width = 16

    ws.row_dimensions[1].height = 48   # header row taller for wrapped CWE names
    ws.freeze_panes = 'B2'             # freeze dataset column + header row

# ── Legend sheet ────────────────────────────────────────────────────────────
leg = wb.create_sheet(title='Legend')
legend_rows = [
    ('Colour',         'Meaning'),
    ('Green (#92D050)', 'Leaf CWE — most specific; no children in MITRE tree'),
    ('Amber (#FFC000)', 'Non-leaf CWE — umbrella / intermediate node; has children'),
    ('Gray  (#BFBFBF)', 'Unknown CWE — ID not found in cwec_latest.xml'),
]
fills = [FILL_HEADER, FILL_LEAF, FILL_NONLEAF, FILL_UNKNOWN]
for r_idx, (col_a, col_b) in enumerate(legend_rows, start=1):
    ca = leg.cell(row=r_idx, column=1, value=col_a)
    cb = leg.cell(row=r_idx, column=2, value=col_b)
    ca.fill = fills[r_idx - 1]
    ca.font = FONT_HEADER if r_idx == 1 else FONT_CWE
    cb.font = FONT_HEADER if r_idx == 1 else FONT_DATA
    ca.alignment = cb.alignment = ALIGN_LEFT
    ca.border = cb.border = BORDER
leg.column_dimensions['A'].width = 22
leg.column_dimensions['B'].width = 60

wb.save(XLSX_PATH)
print(f'Saved: {XLSX_PATH}')
print(f'Sheets: {wb.sheetnames}')

Saved: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\.worktrees\feat-dataset-ingestion\outputs\stage1_dataset_stats.xlsx
Sheets: ['C_C++', 'Legend']


## 6. Summary printout

In [7]:
leaf_cwes    = [c for c in ALL_CWES if cwe_label(c) == 'leaf']
nonleaf_cwes = [c for c in ALL_CWES if cwe_label(c) == 'non-leaf']
unknown_cwes = [c for c in ALL_CWES if cwe_label(c) == 'unknown']

print(f'CWE granularity breakdown across all datasets:')
print(f'  Leaf     (green): {len(leaf_cwes):>3}  — {leaf_cwes[:8]} ...')
print(f'  Non-leaf (amber): {len(nonleaf_cwes):>3}  — {nonleaf_cwes[:8]} ...')
print(f'  Unknown  (gray):  {len(unknown_cwes):>3}  — {unknown_cwes[:8]}')

if nonleaf_cwes:
    print()
    print('Non-leaf CWEs in detail (umbrella nodes — consider mapping to children):')
    for cwe in nonleaf_cwes:
        num  = _strip(cwe)
        name = nav.get_element_name(num)
        total_count = global_counter[cwe]
        print(f'  {cwe:<12} {name:<50} total occurrences: {total_count:,}')

CWE granularity breakdown across all datasets:
  Leaf     (green):  56  — ['CWE-476', 'CWE-416', 'CWE-415', 'CWE-401', 'CWE-617', 'CWE-369', 'CWE-835', 'CWE-122'] ...
  Non-leaf (amber):  95  — ['CWE-119', 'CWE-125', 'CWE-20', 'CWE-787', 'CWE-190', 'CWE-200', 'CWE-399', 'CWE-362'] ...
  Unknown  (gray):  106  — ['NVD-CWE-noinfo', 'CWE-125CWE-787', 'NVD-CWE-Other', 'CWE-Other', 'CWE-20CWE-190', 'CWE-416CWE-362', 'CWE-787CWE-190', 'CWE-787CWE-119']

Non-leaf CWEs in detail (umbrella nodes — consider mapping to children):
  CWE-119      Improper Restriction of Operations within the Bounds of a Memory Buffer total occurrences: 2,066
  CWE-125      Out-of-bounds Read                                 total occurrences: 1,889
  CWE-20       Improper Input Validation                          total occurrences: 1,444
  CWE-787      Out-of-bounds Write                                total occurrences: 1,405
  CWE-190      Integer Overflow or Wraparound                     total occurrences: 875
 